# 02. Geospatial POI Catchment Enrichment & Gold Master Fusion

**Purpose:** This notebook governs the extraction, integration, and engineering of external spatial features derived from OpenStreetMap (OSM) to capture localized demand drivers for 20,000 commercial outlets. 

### Core Engineering Milestones:
1. **Infrastructure Pivot:** Migration from a highly throttled, point-by-point network polling loop inside interactive notebook cells to a dedicated, vectorized, tile-based standalone runner script (`src/features/scrape_poi_features.py`).
2. **Taxonomy Expansion:** Upgrading the spatial feature space from 3 baseline types to 11 complex domain categories mapped at both $500\text{m}$ and $1000\text{m}$ catchment buffers.
3. **Gold Layer Fusion:** Downstream structural grouping of behavioral transactional metrics integrated with external geographical demand scores into a pristine, model-ready Master Feature Table.

In [1]:
import os
import sys
import time
from pathlib import Path
import pandas as pd
import numpy as np

# Establish clear project paths for relative modular imports
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
SILVER_DIR = PROJECT_ROOT / "data" / "silver"
GOLD_DIR = PROJECT_ROOT / "data" / "gold"

print(f"✅ Environment initialized.")
print(f"Silver Source Directory: {SILVER_DIR}")
print(f"Gold Target Directory:   {GOLD_DIR}")

✅ Environment initialized.
Silver Source Directory: /Users/judithfernando/DataStorm7/Repo/datastorm7-jarvis/data/silver
Gold Target Directory:   /Users/judithfernando/DataStorm7/Repo/datastorm7-jarvis/data/gold


### 1. Architectural Alignment & Infrastructure Log

During early operational stress-testing of the Overpass API inside interactive notebook environments, the pipeline encountered chronic `429 Too Many Requests` rate limiting and a complete Google Colab browser session `reCAPTCHA` termination at index 12,500. 

**Resolution:** Rather than accepting feature loss or continuing with fragile network loops, the codebase was refactored into a high-performance standalone engine located at `src/features/scrape_poi_features.py`. 

This specialized script optimized retrieval via:
* **Bounding-Box Tiling:** Aggregating queries into $0.25^\circ \times 0.25^\circ$ tiles to process outlets in spatial blocks instead of unique requests.
* **Vectorized Distance Mapping:** Utilizing a `scikit-learn` `BallTree` spatial index running on radians to map pairwise haversine distances in linear time.

The production extraction completed flawlessly, saving the extracted spatial drivers into `data/silver/outlet_poi_features_FINAL.csv`. We load this validated file directly below to proceed with the engineering pipeline.

In [2]:
# Load the preprocessed Silver Core and the completed V2 Geospatial Feature Layer
master_path = SILVER_DIR / "outlet_master.csv"
tx_path = SILVER_DIR / "transactions_history_final.csv"
poi_path = SILVER_DIR / "outlet_poi_features_FINAL.csv"

# Verify file presence before execution to maintain absolute reproducibility
for path in [master_path, tx_path, poi_path]:
    if not path.exists():
        raise FileNotFoundError(f"Missing mandatory upstream pipeline asset: {path}")

df_master = pd.read_csv(master_path)
df_tx = pd.read_csv(tx_path)
df_poi = pd.read_csv(poi_path)

print(f"📦 Upstream datasets successfully synchronized:")
print(f"  -> Outlet Master Base:      {df_master.shape[0]} outlets")
print(f"  -> Transaction Records:     {df_tx.shape[0]} historical entries")
print(f"  -> Geospatial POI Features: {df_poi.shape[0]} outlets mapped")

📦 Upstream datasets successfully synchronized:
  -> Outlet Master Base:      20000 outlets
  -> Transaction Records:     2339409 historical entries
  -> Geospatial POI Features: 19960 outlets mapped


### 2. Behavioral Feature Extraction (Transaction Profiling)

To collapse millions of rows of transaction histories into a single profile vector per unique store (matching our 1-row-per-outlet modeling requirement), we aggregate lifetime sales patterns, revenue metrics, and order consistency.

In [3]:
print("📊 Processing and aggregating behavioral transaction streams...")

# Compute lifetime sales metrics grouped by unique Outlet_ID
df_tx_profiles = df_tx.groupby('Outlet_ID').agg(
    lifetime_volume_liters=('Volume_Liters', 'sum'),
    lifetime_revenue_lkr=('Total_Bill_Value', 'sum'),
    avg_transaction_size_liters=('Volume_Liters', 'mean'),
    active_transaction_months=('Month', 'count')
).reset_index()

# Merge structural characteristics with transaction profiles via an explicit Left Join
df_gold_base = pd.merge(df_master, df_tx_profiles, on='Outlet_ID', how='left')

# Impute exactly 0 values for newly onboarded outlets that feature no historical transactions
behavioral_cols = ['lifetime_volume_liters', 'lifetime_revenue_lkr', 'avg_transaction_size_liters', 'active_transaction_months']
df_gold_base[behavioral_cols] = df_gold_base[behavioral_cols].fillna(0)

print(f"✅ Behavioral profiling completed. Base Matrix dimensions: {df_gold_base.shape}")

📊 Processing and aggregating behavioral transaction streams...
✅ Behavioral profiling completed. Base Matrix dimensions: (20000, 11)


### 3. Spatial-Relational Fusion & Cross-Layer Imputation

Next, we fuse our internal store matrix with the external 11-category geospatial layer. 

**Data Hygiene Guardrail:** Outlets quarantined during Member 1's data forensics phase due to unresolvable, corrupted coordinate strings must be protected against `NaN` propagation. We apply a robust median imputation strategy to preserve row integrity without distorting data variance.

In [4]:
print("🔗 Executing multi-modal integration between Silver profile matrix and Geospatial layer...")

# Perform left join to anchor our dataset strictly to the 20,000 master retail nodes
df_gold_merged = pd.merge(df_gold_base, df_poi, on='Outlet_ID', how='left')

# Dynamically discover all unique spatial feature metrics generated by our V2 scraper
poi_feature_cols = [col for col in df_poi.columns if col != 'Outlet_ID']

# Apply statistical median imputation over quarantined spatial nodes to ensure zero NaN values
imputation_count = 0
for col in poi_feature_cols:
    null_mask = df_gold_merged[col].isna()
    if null_mask.any():
        imputation_count += null_mask.sum()
        median_metric = df_gold_merged[col].median()
        df_gold_merged[col] = df_gold_merged[col].fillna(median_metric)

print(f"🛡️ Fault-Recovery Complete: Handled {imputation_count} missing cell features via global median values.")

🔗 Executing multi-modal integration between Silver profile matrix and Geospatial layer...
🛡️ Fault-Recovery Complete: Handled 1080 missing cell features via global median values.


### 4. Advanced Interaction Feature Engineering

Instead of passing raw baseline counts directly into the model, we engineer domain-specific spatial indices that serve as functional indicators of market characteristics:
* **`transit_hub_connectivity_index`**: Tracks immediate logistical foot-traffic density using bus and railway hubs.
* **`commercial_velocity_ratio`**: Computes the ratio of immediate supermarkets/markets ($500\text{m}$) relative to their wider local baseline ($1000\text{m}$) to measure competitive local cluster density.

In [5]:
print("🧠 Constructing advanced latent interaction metrics...")

# Logistical Transit Connectivity Feature
df_gold_merged['transit_hub_connectivity_index'] = (
    df_gold_merged['bus_stops_500m'] + (df_gold_merged['railway_stations_500m'] * 2.5)
)

# Commercial Velocity Density Ratio (safely avoiding zero-division bottlenecks)
df_gold_merged['commercial_velocity_ratio'] = (
    (df_gold_merged['supermarkets_500m'] + df_gold_merged['markets_500m'] + 1) /
    (df_gold_merged['supermarkets_1000m'] + df_gold_merged['markets_1000m'] + 1)
)

print("✅ Interaction features successfully instantiated.")

🧠 Constructing advanced latent interaction metrics...
✅ Interaction features successfully instantiated.


### 5. Final Enterprise Validation & Gold Promotion

Prior to exporting this master asset to modeling environments, we run strict quality constraints to guarantee pipeline consistency.

In [6]:
# 1. Structural Integrity Check
assert len(df_gold_merged) == 20000, f"CRITICAL ERROR: Row dimensions altered! Length: {len(df_gold_merged)}"

# 2. Total Missing Value Matrix Sweep
total_nans = df_gold_merged.isna().sum().sum()
assert total_nans == 0, f"CRITICAL ERROR: {total_nans} unhandled missing cells found in final matrix!"

# 3. Export to Production Path
GOLD_DIR.mkdir(parents=True, exist_ok=True)
output_file_path = GOLD_DIR / "master_feature_table.csv"
df_gold_merged.to_csv(output_file_path, index=False)

print("🏆 GOLD LAYER FUSION PIPELINE COMPLETE!")
print(f"Master Feature Table pushed to: {output_file_path}")
print(f"Final Matrix Footprint:         {df_gold_merged.shape[0]} outlets x {df_gold_merged.shape[1]} columns")

print("\n📋 Sample Vector Preview:")
display(df_gold_merged.head(3))

🏆 GOLD LAYER FUSION PIPELINE COMPLETE!
Master Feature Table pushed to: /Users/judithfernando/DataStorm7/Repo/datastorm7-jarvis/data/gold/master_feature_table.csv
Final Matrix Footprint:         20000 outlets x 40 columns

📋 Sample Vector Preview:


,Outlet_ID,Outlet_Size,Cooler_Count,Outlet_Type,outlet_size_status,coord_status,has_valid_coord,lifetime_volume_liters,lifetime_revenue_lkr,avg_transaction_size_liters,...,tourism_1000m,railway_stations_500m,railway_stations_1000m,total_poi_500m,total_poi_1000m,demand_driver_score_500m,demand_driver_score_1000m,poi_available,transit_hub_connectivity_index,commercial_velocity_ratio
0,OUT_00001,Medium,1,Grocery,provided,valid,True,12287.669769,3.152152e+06,64.671946,...,0.0,0.0,0.0,8.0,24.0,7.6,22.8,True,0.0,0.6
1,OUT_00002,Small,0,Hotel,provided,valid,True,14494.419585,3.703930e+06,71.754552,...,0.0,0.0,0.0,0.0,2.0,0.0,1.7,True,0.0,1.0
2,OUT_00003,Small,1,Pharmacy,provided,valid,True,12620.115111,3.515750e+06,60.673630,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,False,0.0,1.0
